In [2]:
library("lme4")
library("margins")
library("stargazer")
library("emmeans")
library("ggeffects")
library("broom")
library("broom.mixed")
library("MASS")
library("pscl")
library("fixest")
library("marginaleffects")
library("modelsummary")
library("glmmTMB")
library("dplyr")

In [3]:
main_path <- "/home/20250114zmz_kd/"
data <- read.csv(paste0(main_path, "UseBSForNot/MergeInsthData_1211.csv"))
dim(data)

[1] 3227892      78

In [4]:
print(names(data))

 [1] "X"                                           
 [2] "work_id"                                     
 [3] "fac_pub"                                     
 [4] "year"                                        
 [5] "paper_type"                                  
 [6] "paper_language"                              
 [7] "nwbin"                                       
 [8] "new_word_reuse_times"                        
 [9] "npbin"                                       
[10] "new_phrase_reuse_times"                      
[11] "novelbin"                                    
[12] "rao_stirling"                                
[13] "author_id"                                   
[14] "author_position"                             
[15] "institution_id"                              
[16] "country_code"                                
[17] "country"                                     
[18] "num_author_log"                              
[19] "num_inst_log"                                
[20] "num_co

In [5]:
colSums(is.na(data))

X 
                                           0 
                                     work_id 
                                           0 
                                     fac_pub 
                                           0 
                                        year 
                                           0 
                                  paper_type 
                                           0 
                              paper_language 
                                           0 
                                       nwbin 
                                           0 
                        new_word_reuse_times 
                                           0 
                                       npbin 
                                           0 
                      new_phrase_reuse_times 
                                           0 
                                    novelbin 
                                           0 
                                rao_stirling 
                                      303430 
                                   author_id 
                                           0 
                             author_position 
                                           0 
                              institution_id 
                                           0 
                                country_code 
                                           0 
                                     country 
                                           0 
                              num_author_log 
                                           0 
                                num_inst_log 
                                           0 
                             num_country_log 
                                           0 
                           num_reference_log 
                                           0 
                             timescited5_log 
                                      107147 
                            timescited10_log 
                                      163886 
                           timescitedall_log 
                                           0 
                               ab_length_log 
                                           0 
                     leadership_global_class 
                                           0 
                         mean_career_age_log 
                                           0 
                            inst_h_index_log 
                                           6 
                                   source_id 
                                           0 
                          source_h_index_log 
                                           0 
                                 source_type 
                                           0 
                                 core_source 
                                           0 
        Agricultural.and.Biological.Sciences 
                                           0 
                         Arts.and.Humanities 
                                           0 
Biochemistry..Genetics.and.Molecular.Biology 
                                           0 
         Business..Management.and.Accounting 
                                           0 
                        Chemical.Engineering 
                                           0 
                                   Chemistry 
                                           0 
                            Computer.Science 
                                           0 
                           Decision.Sciences 
                                           0 
                                   Dentistry 
                                           0 
                Earth.and.Planetary.Sciences 
                                           0 
         Economics..Econometrics.and.Finance 
                                           0 
                                      Energy 
                                         

In [6]:
# 把所有无限值替换成 NA
data[sapply(data, is.infinite)] <- NA

In [7]:
data <- data[!is.na(data$new_word), ]
dim(data)
# data <- data[data$num_reference >=5, ]
# data <- data[data$timescited10 >=5, ]
dim(data)

[1] 2864380      78

[1] 2864380      78

In [8]:
data <- data[data$ab_length > 10, ]
dim(data)
data <- data[data$year >= 1950, ]
# data <- data[data$year <= 2014, ]
dim(data)

[1] 1875103      78

[1] 1875103      78

In [9]:
# 找出所有包含无限值的行和列
inf_mask <- sapply(data, function(col) is.infinite(col))
rows_with_inf <- apply(inf_mask, 1, any)  # 哪些行至少有一个Inf
cols_with_inf <- colnames(data)[apply(inf_mask, 2, any)]  # 哪些列有Inf

# 打印包含无限值的行数和列名
cat("包含无限值的行数:", sum(rows_with_inf), "\n")
cat("包含无限值的列名:", paste(cols_with_inf, collapse = ", "), "\n")

# 查看这些行具体内容
data_filt_with_inf <- data[rows_with_inf, c(cols_with_inf), drop=FALSE]
print(data_filt_with_inf)

包含无限值的行数: 0 
包含无限值的列名:  
data frame with 0 columns and 0 rows


In [10]:
data$leadership_global_class <- factor(data$leadership_global_class)
data <- within(data, leadership_global_class <- relevel(leadership_global_class, ref = 'shared'))
data$fac_pub <- factor(data$fac_pub)
data <- within(data, fac_pub <- relevel(fac_pub, ref = 'NonBSF'))
data$core_source <- factor(data$core_source)
data <- within(data, core_source <- relevel(core_source, ref = 'NonCore'))
data$year <- as.factor(data$year)

In [11]:
# variable type
paper_level <- "num_author_log + num_inst_log + num_country_log + num_reference_log + leadership_global_class + mean_career_age_log + inst_h_index_log + ab_length_log"
journal_level <- "source_h_index_log + core_source"
disciplines <- "Arts.and.Humanities + Biochemistry..Genetics.and.Molecular.Biology + Business..Management.and.Accounting + Chemical.Engineering + 
 Chemistry + Computer.Science + Decision.Sciences + Dentistry + Earth.and.Planetary.Sciences + 
Economics..Econometrics.and.Finance + Energy + Engineering + Environmental.Science + Health.Professions + 
Immunology.and.Microbiology + Materials.Science + Mathematics + Medicine + Neuroscience + Nursing +
Pharmacology..Toxicology.and.Pharmaceutics + Physics.and.Astronomy + Psychology + Social.Sciences + Veterinary "

In [12]:
fml <- as.formula(
  paste0("nwbin ~ fac_pub + ", paper_level, "+", journal_level, "+", disciplines, " | author_id + year")
)

model_total <- feglm(fml, data = data, family = binomial("logit"), vcov = "iid")
summary(model_total)

NOTES: 3 observations removed because of NA values (RHS: 3).
       35,590/1 fixed-effects (254,649 observations) removed because of only 0 (or only 1) outcomes.



GLM estimation, family = binomial, Dep. Var.: nwbin
Observations: 1,620,451
Fixed-effects: author_id: 39,257,  year: 74
Standard-errors: IID 
                                              Estimate Std. Error    z value
fac_pubBSF                                    0.049569   0.012090   4.100141
num_author_log                                0.303582   0.008518  35.638712
num_inst_log                                 -0.064563   0.011289  -5.718836
num_country_log                               0.034774   0.016938   2.053023
num_reference_log                            -0.020506   0.002887  -7.103358
leadership_global_classallNorth               0.001414   0.019497   0.072523
leadership_global_classallSouth               0.011529   0.028845   0.399703
mean_career_age_log                          -0.048076   0.008894  -5.405195
inst_h_index_log                             -0.020054   0.006645  -3.017951
ab_length_log                                 0.311117   0.004636  67.114747
source_h_in

In [13]:
paper_vars <- c("num_author_log", "num_inst_log", "num_country_log",
                "num_reference_log", "leadership_global_class",
                "mean_career_age_log", "inst_h_index_log" )


journal_vars <- c("source_h_index_log", "core_source")

disciplines_vars <- c("Arts.and.Humanities", "Biochemistry..Genetics.and.Molecular.Biology", "Business..Management.and.Accounting",
                 "Chemical.Engineering", "Chemistry", "Computer.Science", "Decision.Sciences", "Dentistry",
                 "Earth.and.Planetary.Sciences", "Economics..Econometrics.and.Finance", "Energy", "Engineering",
                 "Environmental.Science + Health.Professions", "Immunology.and.Microbiology", "Materials.Science", "Mathematics",
                 "Medicine", "Neuroscience", "Nursing", "Pharmacology..Toxicology.and.Pharmaceutics", "Physics.and.Astronomy",
                 "Psychology", "Social.Sciences", "Veterinary")

In [14]:
etable_list <- vector("list", 1) 

etable_list[[1]] <- etable(model_total,
                           keep = c("fac_pub", paper_vars, journal_vars),
                           se = "iid",
                           # cluster = ~ paper_year + paper_field,
                           tex = TRUE,
                           digits = 3)
# 合并 LaTeX 代码（手动拼接）
tab_latex <- paste(unlist(etable_list), collapse = "\n")
cat(tab_latex)

\begingroup
\centering
\begin{tabular}{lc}
   \tabularnewline \midrule \midrule
   Dependent Variable:                 & nwbin\\  
   Model:                              & (1)\\  
   \midrule
   \emph{Variables}\\
   fac\_pubBSF                         & 0.050$^{***}$\\   
                                       & (0.012)\\   
   num\_author\_log                    & 0.304$^{***}$\\   
                                       & (0.009)\\   
   num\_inst\_log                      & -0.065$^{***}$\\   
                                       & (0.011)\\   
   num\_country\_log                   & 0.035$^{**}$\\   
                                       & (0.017)\\   
   num\_reference\_log                 & -0.021$^{***}$\\   
                                       & (0.003)\\   
   leadership\_global\_classallNorth   & 0.001\\   
                                       & (0.019)\\   
   leadership\_global\_classallSouth   & 0.011\\   
                                       & (0.029)\\   
   

In [15]:
# 每组 reg_class 的平均预测概率
margins_eff_reg_class <- avg_predictions(model_total, variables = "fac_pub")
margins_eff_reg_class

fac_pub,estimate,std.error,statistic,p.value,s.value,conf.low,conf.high
<fct>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
NonBSF,0.1004705,0.005115019,19.64225,6.734531e-86,282.9342,0.09044523,0.1104957
BSF,0.1046544,0.005309232,19.71178,1.708526e-86,284.9131,0.09424851,0.1150603


In [16]:
fname = paste0(main_path, "UseBSForNot/predict_nw_1211.csv")
write.csv(margins_eff_reg_class, fname, row.names = FALSE)